# Taxonomic Relevance Evaluation

This notebook evaluates a problem in the current species scoring setup: the manual annotations often encode dataset relevance through coarse taxonomic group and richness information, while the extraction output often enumerates detailed taxa. That creates large numbers of false positives and false negatives even when the extraction is arguably useful for screening dataset relevance.

The working hypothesis tested here is that we should compare several parallel taxonomic views instead of relying on the raw `species` field alone:

- raw `species` strings with the existing enhanced matcher,
- derived `taxon_richness_counts` to compare count-bearing GT strings against enumerated predictions,
- derived `taxon_richness_group_keys` to compare count + normalized group labels,
- derived `taxon_broad_group_labels` to compare broader group labels inferred from explicit mentions and GBIF hierarchy,
- derived `gbif_keys` to collapse vernacular vs scientific naming.


## Questions

1. Which mismatch patterns are evaluation artifacts rather than extraction failures?
2. Does a count-focused derived field recover signal from cases like `73 weevil species` vs 73 enumerated taxa?
3. Does GBIF enrichment improve comparison for vernacular/scientific mismatches?
4. Which derived fields reflect dataset relevance better than the baseline `species` metric?


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from llm_metadata.schemas.data_paper import RunArtifact
from llm_metadata.schemas.fuster_features import DatasetFeatures
from llm_metadata.taxonomy_eval import (
    DEFAULT_TAXONOMY_FIELDS,
    build_ground_truth_by_id,
    enrich_indexed_models,
    evaluate_taxonomy_fields,
)


In [2]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RUN_GLOB = "*20260305_135009_dev_subset_pdf_file*.json"
GT_PATH = PROJECT_ROOT / "data/dataset_092624_validated.xlsx"

candidates = sorted((PROJECT_ROOT / "artifacts/runs").glob(RUN_GLOB))
candidates += sorted((PROJECT_ROOT / "data/prompt_eval_reports").glob(RUN_GLOB))
RUN_ARTIFACT_PATH = candidates[0] if candidates else PROJECT_ROOT / "data/prompt_eval_reports/20260305_135009_dev_subset_pdf_file.json"

if not RUN_ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find run artifact. Update RUN_GLOB or RUN_ARTIFACT_PATH. Tried: {RUN_ARTIFACT_PATH}"
    )

try:
    artifact = RunArtifact.load_json(RUN_ARTIFACT_PATH)
    allowed_ids = [record.gt_record_id for record in artifact.records]
    pred_by_id = artifact.predictions_by_id(DatasetFeatures)
except Exception:
    artifact = None
    raw_report = json.loads(RUN_ARTIFACT_PATH.read_text(encoding="utf-8"))
    allowed_ids = [int(record_id) for record_id in raw_report["records"].keys()]
    pred_rows = {}
    for row in raw_report["field_results"]:
        record_id = str(row["record_id"])
        pred_rows.setdefault(record_id, {})[row["field"]] = row["pred_value"]
    pred_by_id = {
        record_id: DatasetFeatures.model_validate(values)
        for record_id, values in pred_rows.items()
    }

true_by_id = build_ground_truth_by_id(GT_PATH, allowed_ids=allowed_ids)

display(Markdown(
    f"**Run artifact:** `{RUN_ARTIFACT_PATH}`  \n**Records loaded:** GT={len(true_by_id)} | Pred={len(pred_by_id)}"
))


**Run artifact:** `C:\Users\beav3503\dev\llm_metadata\data\prompt_eval_reports\20260305_135009_dev_subset_pdf_file.json`  
**Records loaded:** GT=30 | Pred=30

## Proposed Solution Under Test

The proposal is not to replace the raw `species` field. It is to keep it, then derive additional taxonomic comparison views from the same `DatasetFeatures` objects:

- `parsed_species`: structured analysis view only,
- `taxon_richness_mentions`: structured count + group mentions,
- `taxon_richness_counts`: exact comparison on projected counts,
- `taxon_richness_group_keys`: exact comparison on projected `count|group` keys,
- `taxon_broad_group_labels`: exact comparison on broader relevance-oriented group labels,
- `gbif_keys`: exact comparison on resolved taxon identifiers.

This keeps the extraction contract stable while making evaluation more aligned with the annotation intent.


In [3]:
FIELDS = [
    "species",
    "taxon_richness_counts",
    "taxon_richness_group_keys",
    "taxon_broad_group_labels",
    "gbif_keys",
]

# Set include_gbif=False if you want to avoid live GBIF lookups or rely only on cached calls.
report = evaluate_taxonomy_fields(
    true_by_id=true_by_id,
    pred_by_id=pred_by_id,
    fields=FIELDS,
    include_gbif=True,
    gbif_confidence_threshold=80,
)

metrics_df = report.metrics_to_pandas().sort_values(["f1", "precision", "recall"], ascending=False)
metrics_df


,field,tp,fp,fn,tn,n,precision,recall,f1,accuracy,exact_match_rate
2,taxon_broad_group_labels,13,16,6,7,30,0.448276,0.684211,0.541667,0.666667,0.566667
3,taxon_richness_counts,16,14,16,0,30,0.533333,0.500000,0.516129,0.533333,0.533333
1,species,36,244,16,0,30,0.128571,0.692308,0.216867,1.200000,0.466667
0,gbif_keys,19,246,2,0,30,0.071698,0.904762,0.132867,0.633333,0.333333
4,taxon_richness_group_keys,0,3,12,19,30,0.000000,0.000000,NaN,0.633333,0.633333


In [4]:
true_enriched = enrich_indexed_models(true_by_id, include_gbif=False)
pred_enriched = enrich_indexed_models(pred_by_id, include_gbif=False)
detail_df = report.to_pandas()

species_mismatches = detail_df[(detail_df["field"] == "species") & (~detail_df["match"])]
richness_mismatches = detail_df[(detail_df["field"] == "taxon_richness_counts") & (~detail_df["match"])]
group_mismatches = detail_df[(detail_df["field"] == "taxon_richness_group_keys") & (~detail_df["match"])]
broad_group_mismatches = detail_df[(detail_df["field"] == "taxon_broad_group_labels") & (~detail_df["match"])]

display(Markdown("### Baseline species mismatches"))
display(species_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(20))

display(Markdown("### Count-based mismatches"))
display(richness_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(20))

display(Markdown("### Count+group mismatches"))
display(group_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(20))

display(Markdown("### Broad-group mismatches"))
display(broad_group_mismatches[["record_id", "true_value", "pred_value", "tp", "fp", "fn"]].head(20))


### Baseline species mismatches

,record_id,true_value,pred_value,tp,fp,fn
21,123,[Caribou],"[Rangifer tarandus, migratory tundra caribou, ...",1,1,0
26,175,"[sapin baumier, epinette blanche, bouleau jaun...","[Abies balsamea (L.) Mill., Betula alleghanien...",5,12,1
41,208,[c.132 species of benthic community],"[132 taxa (8 phyla) — dominance of arthropods,...",0,19,1
46,225,"[Benthic intertidal community (c.20 algae, c.1...","[Fucus distichus edentatus, Fucus vesiculosus,...",0,10,4
51,24,[Acer saccharum],"[Acer saccharum Marshall (sugar maple), 953 in...",1,1,0
61,258,"[16 damselfly species, water mite]","[Amphiagrion saucium, Chromagrion conditum, Co...",1,16,1
66,27,"[Eastern coyote, eastern wolf]","[Canis lycaon (Eastern Wolf), Canis latrans (E...",2,2,0
71,271,[11 mite species],"[Amblyseius fallacis (Garman), Typhlodromus ca...",0,13,1
76,29,"[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...",4,2,0
81,30,[Rangifer tarandus],[Rangifer tarandus caribou (Rivière-aux-Feuill...,1,1,0


### Count-based mismatches

,record_id,true_value,pred_value,tp,fp,fn
3,101,[1],[49],0,1,1
23,123,[1],[4],0,1,1
28,175,[6],[17],0,1,1
48,225,[15],[10],0,1,1
53,24,[1],[953],0,1,1
63,258,[16],[17],0,1,1
68,27,[2],[4],0,1,1
73,271,[11],[13],0,1,1
78,29,[4],[6],0,1,1
83,30,[1],[2],0,1,1


### Count+group mismatches

,record_id,true_value,pred_value,tp,fp,fn
4,101,None,[49|female caribou],0,1,0
44,208,[132|benthic community],"[132|— dominance of arthropods, mollusks, and ...",0,1,1
49,225,"[15|annelid, 15|mollusc]",None,0,0,2
54,24,None,[953|individuals sampled; 913 individuals geno...,0,1,0
64,258,[16|damselfly],None,0,0,1
74,271,[11|mite],None,0,0,1
94,315,[30|beetle],None,0,0,1
99,343,[30|bird],None,0,0,1
104,351,[73|weevil],None,0,0,1
114,38,"[12|mammal, 199|ground-dwelling beetle, 240|fl...",None,0,0,3


### Broad-group mismatches

,record_id,true_value,pred_value,tp,fp,fn
2,101,None,"[female caribou, mammal]",0,2,0
7,104,None,[mammal],0,1,0
22,123,None,[mammal],0,1,0
32,19,None,[tick],0,1,0
42,208,[benthic community],"[annelid, arthropod, mollusc, — dominance of a...",0,4,1
47,225,"[annelid, mollusc]","[arthropod, mollusc]",1,1,1
52,24,None,[individuals sampled; 913 individuals genotype...,0,1,0
62,258,[damselfly],"[arthropod, mite]",0,2,1
67,27,None,[mammal],0,1,0
102,351,[weevil],"[beetle, weevil]",1,1,0


In [5]:
records = []
for record_id in sorted(set(true_enriched) | set(pred_enriched), key=int):
    true_model = true_enriched.get(record_id)
    pred_model = pred_enriched.get(record_id)
    records.append(
        {
            "record_id": record_id,
            "true_species": getattr(true_model, "species", None),
            "pred_species": getattr(pred_model, "species", None),
            "true_richness_mentions": getattr(true_model, "taxon_richness_mentions", None),
            "pred_richness_mentions": getattr(pred_model, "taxon_richness_mentions", None),
            "true_richness_counts": getattr(true_model, "taxon_richness_counts", None),
            "pred_richness_counts": getattr(pred_model, "taxon_richness_counts", None),
            "true_group_keys": getattr(true_model, "taxon_richness_group_keys", None),
            "pred_group_keys": getattr(pred_model, "taxon_richness_group_keys", None),
            "true_broad_groups": getattr(true_model, "taxon_broad_group_labels", None),
            "pred_broad_groups": getattr(pred_model, "taxon_broad_group_labels", None),
        }
    )

taxonomy_view_df = pd.DataFrame.from_records(records)
taxonomy_view_df.head(20)


,record_id,true_species,pred_species,true_richness_mentions,pred_richness_mentions,true_richness_counts,pred_richness_counts,true_group_keys,pred_group_keys,true_broad_groups,pred_broad_groups
0,5,[Glyptemys insculpta],[Glyptemys insculpta],None,None,[1],[1],None,None,None,None
1,9,"[raccoons, striped skunks]","[Procyon lotor (raccoon), Mephitis mephitis (s...",None,None,[2],[2],None,None,None,None
2,11,[Rangifer tarandus caribou],[Rangifer tarandus caribou (woodland caribou)],None,None,[1],[1],None,None,None,None
3,12,[Rangifer tarandus caribou],[Rangifer tarandus caribou (woodland caribou)],None,None,[1],[1],None,None,None,None
4,19,[black-legged tick],[Ixodes scapularis Say 1821 (black-legged tick)],None,None,[1],[1],None,None,None,None
5,24,[Acer saccharum],"[Acer saccharum Marshall (sugar maple), 953 in...",None,[original='953 individuals sampled; 913 indivi...,[1],[953],None,[953|individuals sampled; 913 individuals geno...,None,[individuals sampled; 913 individuals genotype...
6,27,"[Eastern coyote, eastern wolf]","[Canis lycaon (Eastern Wolf), Canis latrans (E...",None,None,[2],[4],None,None,None,None
7,29,"[Rhododendron groenlandicum, Kalmia angustifol...","[Rhododendron groenlandicum, Kalmia angustifol...",None,None,[4],[6],None,None,None,None
8,30,[Rangifer tarandus],[Rangifer tarandus caribou (Rivière-aux-Feuill...,None,None,[1],[2],None,None,None,None
9,31,[Atlantic salmon],[Atlantic salmon (Salmo salar)],None,None,[1],[1],None,None,None,None


## Discussion Guide

Use the outputs above to answer these points explicitly:

- Which baseline `species` mismatches become reasonable matches under `taxon_richness_counts`?
- Does `taxon_broad_group_labels` recover useful signal from enumerated predictions that never emit an explicit coarse group string?
- Which cases remain unresolved because group identity is still too coarse or too specific?
- How often does `gbif_keys` solve scientific vs vernacular mismatch without helping count/group cases?
- Do the new fields better reflect dataset relevance, or are they masking real extraction errors?

Alternatives to assess if the derived fields are still insufficient:

- prompt the model to extract structured taxon objects directly,
- refine the GBIF-to-group mapping if the current broad-group derivation is still too sparse or too permissive,
- add a small manual adjudication subset to validate that the new metrics really track relevance.
